In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file



True

In [2]:
from google import genai

client = genai.Client(api_key=os.environ.get("GOOGLE_GENAI_API_KEY"))
def get_gemini_response(prompt: str):
    response = client.models.generate_content(
        model="gemini-2.5-pro", contents=prompt
    )
    return response.text


In [3]:
from __future__ import annotations

import json
import os
import re
import tempfile
from typing import Dict, List
import pandas as pd
from tqdm import tqdm
from pathlib import Path

SUPPORTED_LANGUAGES = [
    "english", "german", "swedish", "latin", "spanish", "chinese", "norwegian_1", "norwegian_2"
]

TEMPLATES: Dict[str, str] = {
    "english": 'Write all dictionary definitions of "{WORD}" in English.\n'
               "One sense per line. Output only the definitions. Do not add any text before or after the definitions.",
    "german":  'Schreibe alle Wörterbuchdefinitionen von "{WORD}" auf Deutsch.\n'
               "Eine Bedeutung pro Zeile. Gib nur die Definitionen aus. Füge keinen Text vor oder nach den Definitionen hinzu.",
    "swedish": 'Skriv alla ordboksdefinitioner av "{WORD}" på svenska.\n'
               "En betydelse per rad. Skriv endast definitionerna. Lägg inte till någon text före eller efter definitionerna.",
    "latin":   'Scribe omnes definitiones dictionarii verbi "{WORD}" Latine.\n'
               "Una significatio per lineam. Redde tantum definitiones. Ne quidquam addas ante aut post definitiones.",
    "spanish": 'Escribe todas las definiciones de diccionario de "{WORD}" en español.\n'
               "Un significado por línea. Devuelve solo las definiciones. No añadas ningún texto antes ni después de las definiciones.",
    "chinese": '用中文写出“{WORD}”的所有词典释义。\n'
               "每行一个义项。只输出释义，不要在释义前后添加任何文字。",
    "norwegian_1": 'Skriv alle ordbokdefinisjoner av "{WORD}" på norsk.\n'
                   "Én betydning per linje. Svar kun med definisjonene. Ikke legg til tekst før eller etter definisjonene.",
    "norwegian_2": 'Skriv alle ordbokdefinisjoner av "{WORD}" på norsk.\n'
                   "Én betydning per linje. Svar kun med definisjonene. Ikke legg til tekst før eller etter definisjonene.",
}

def write_prompt(language: str, word: str) -> str:
    return TEMPLATES[language.lower().strip()].format(WORD=word)

def short_model(model: str) -> str:
    s = model.lower().strip().replace(":", "_")
    s = re.sub(r"[^a-z0-9._-]+", "", s).replace(".", "").replace("-", "")
    return s

_BULLET = re.compile(r"^\s*(?:[-•*]+|\(?\d+\)?[.)]|[a-zA-Z][.)])\s+")
_SP = re.compile(r"\s+")

def parse_defs(text: str) -> List[str]:
    out: List[str] = []
    for line in text.splitlines():
        line = _SP.sub(" ", _BULLET.sub("", line.strip())).strip()
        if line:
            out.append(line)
    return out

def _atomic_json_write(path: str, obj: dict) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    fd, tmp = tempfile.mkstemp(dir=os.path.dirname(path) or ".", prefix=".tmp_", suffix=".json")
    with os.fdopen(fd, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, sort_keys=True)
    os.replace(tmp, path)

def append_definitions(language: str, model: str, word: str, model_output_text: str, out_dir: str = ".") -> str:
    lang = language.lower().strip()
    path = os.path.join(out_dir, f"{lang}_definitions_by_{short_model(model)}.json")

    defs = parse_defs(model_output_text)
    doc = json.load(open(path, "r", encoding="utf-8")) if os.path.exists(path) else {}

    # Overwrite this word only (do not touch other words)
    doc[word] = defs

    _atomic_json_write(path, doc)
    return path

def create_df_from_graded_changes(path_to_stats):
    df = pd.read_csv(path_to_stats, delimiter='\t')
    keep_df = df[['lemma','change_graded','change_binary']]
    keep_df = keep_df.rename(columns={'lemma':'words', 'change_graded':'graded_scores', 'change_binary':'binary_scores'})
    return keep_df


SEMEVAL_DATA_ROOT = os.getenv("SEMEVAL_DATA_ROOT")

def load_semeval_df(lang: str):
    """Return the correct dataframe for the given semeval language.
    These dataframes should be pre-prepared to contain the columns:
    words, graded_scores, binary_scores
    """

    csv_paths = {
        'english': SEMEVAL_DATA_ROOT + '/semeval2020_ulscd_eng/en_definitions.csv',
        'german': SEMEVAL_DATA_ROOT + '/semeval2020_ulscd_ger/de_definitions.csv',
        'swedish': SEMEVAL_DATA_ROOT + '/semeval2020_ulscd_swe/swe_definitions.csv',
        'latin': SEMEVAL_DATA_ROOT + '/semeval2020_ulscd_lat/lat_definitions.csv',
        'spanish': SEMEVAL_DATA_ROOT + '/dwug_es/es_definitions.csv',
    }

    special_handlers = {
        'chinese': lambda: create_df_from_graded_changes(
            SEMEVAL_DATA_ROOT + '/chiwug/stats/opt/stats_groupings.csv'
        ),

        'norwegian_1': lambda: create_df_from_graded_changes(
            SEMEVAL_DATA_ROOT + '/nor_dia_change/subset1/stats/stats_groupings.tsv'
        ).rename(columns={
            'lemma': 'words',
            'change_graded': 'graded_scores',
            'change_binary': 'binary_scores'
        }),

        'norwegian_2': lambda: create_df_from_graded_changes(
            SEMEVAL_DATA_ROOT + '/nor_dia_change/subset2/stats/stats_groupings.tsv'
        ).rename(columns={
            'lemma': 'words',
            'change_graded': 'graded_scores',
            'change_binary': 'binary_scores'
        }),
    }

    if lang in csv_paths:
        return pd.read_csv(csv_paths[lang])

    if lang in special_handlers:
        return special_handlers[lang]()

    raise ValueError(f"Unknown semeval language: {lang}")



In [4]:

langs = ['english','german','swedish','latin','spanish','chinese','norwegian_1','norwegian_2']

lang_words = {}
for lang in langs:
    df = load_semeval_df(lang)
    print(f"{lang} has {len(df)} words.")
    words = df['words'].tolist()
    lang_words[lang] = words

# for every word in english, take the first of the split _
lang_words['english'] = [w.split('_')[0] for w in lang_words['english']]

english has 37 words.
german has 48 words.
swedish has 31 words.
latin has 40 words.
spanish has 100 words.
chinese has 40 words.
norwegian_1 has 40 words.
norwegian_2 has 40 words.


In [ ]:
# Gemini definitions generation
for lang, words in lang_words.items():
    print(f"{lang}: {len(words)} words")
    for word in tqdm(words):
        prompt = write_prompt(lang,word)
        definitions = get_gemini_response(prompt=prompt)
        path = append_definitions(
            language=lang,
            model="gemini-2.5-pro",      
            word=word,
            model_output_text=definitions,
            out_dir="definitions",        # folder to store the json files
    )
    
# 78